# Decision tree — entropy-based attribute selection

This notebook follows the supplied `decisionTree(D, A, T)` pseudocode,
using entropy to measure impurity. Information Gain is the default ranking;
an adjacent commented assignment enables Gain Ratio instead. Training creates
one branch per observed value and removes the selected attribute before recursion.

This is not the complete C4.5 algorithm. Continuous-threshold search,
missing-value weighting, pruning, and input validation are not part of training.

Run the five code cells from top to bottom in a Python 3 notebook kernel.
Only Python's standard library is needed; there is no machine-learning library.


## 1. Input data

Provide a nonempty rectangular list matrix. Its first row contains column names;
each following row is an observation, and the **last column is always the class**.
The header is not passed to training. Attribute values and classes are assumed
to be hashable. The algorithm assumes well-formed input; it does not validate it.

Every attribute is split by its observed values, including numbers. For example,
rows `[1, "no"], [2, "no"], [3, "yes"], [4, "yes"]` produce branches for
`1`, `2`, `3`, and `4`, not a numerical cut at `2.5`.

NumPy/pandas inputs can be converted with `array.tolist()`
or `dataframe.to_numpy().tolist()`, supplying the header separately.


In [6]:
training_matrix = [  # Define D como una matriz con encabezados y la clase en la última columna.
    ["Outlook", "Temperature", "Humidity", "Wind", "Play"],  # Etiqueta las columnas de atributos y la clase objetivo Play.
    ["sunny", "hot", "high", "weak", "no"],  # Ejemplo de entrenamiento: cuatro atributos y su clase conocida.
    ["sunny", "hot", "high", "strong", "no"],  # Ejemplo de entrenamiento: cuatro atributos y su clase conocida.
    ["overcast", "hot", "high", "weak", "yes"],  # Ejemplo de entrenamiento: cuatro atributos y su clase conocida.
    ["rain", "mild", "high", "weak", "yes"],  # Ejemplo de entrenamiento: cuatro atributos y su clase conocida.
    ["rain", "cool", "normal", "weak", "yes"],  # Ejemplo de entrenamiento: cuatro atributos y su clase conocida.
    ["rain", "cool", "normal", "strong", "no"],  # Ejemplo de entrenamiento: cuatro atributos y su clase conocida.
    ["overcast", "cool", "normal", "strong", "yes"],  # Ejemplo de entrenamiento: cuatro atributos y su clase conocida.
    ["sunny", "mild", "high", "weak", "no"],  # Ejemplo de entrenamiento: cuatro atributos y su clase conocida.
    ["sunny", "cool", "normal", "weak", "yes"],  # Ejemplo de entrenamiento: cuatro atributos y su clase conocida.
    ["rain", "mild", "normal", "weak", "yes"],  # Ejemplo de entrenamiento: cuatro atributos y su clase conocida.
    ["sunny", "mild", "normal", "strong", "yes"],  # Ejemplo de entrenamiento: cuatro atributos y su clase conocida.
    ["overcast", "mild", "high", "strong", "yes"],  # Ejemplo de entrenamiento: cuatro atributos y su clase conocida.
    ["overcast", "hot", "normal", "weak", "yes"],  # Ejemplo de entrenamiento: cuatro atributos y su clase conocida.
    ["rain", "mild", "high", "strong", "no"],  # Ejemplo de entrenamiento: cuatro atributos y su clase conocida.
]  # Completa la matriz de ejemplos de entrenamiento.

columns = training_matrix[0]  # Separa los nombres de las columnas, que no son observaciones.
D = training_matrix[1:]  # Obtiene el conjunto D que recibe el algoritmo de la imagen.
A = list(range(len(columns) - 1))  # Forma A con los índices de atributos; excluye la columna de clase.


## 2. Tree structure

A leaf has `attribute=None` and returns `prediction`. An internal node stores
an attribute's column index and children keyed by its observed values.
The optional `Node.threshold` field remains available to the existing standalone
prediction/display helpers, but literal training always leaves it as `None`.

Every node keeps its training majority class for the unseen-value fallback
implemented in the separate `predict_one()` helper.
A class-frequency tie uses the first class encountered.


In [7]:
class Node:  # Define la estructura del nodo T del pseudocódigo.
    def __init__(self, prediction, attribute=None, threshold=None):  # Inicializa un nodo como hoja; luego puede convertirse en nodo de decisión.
        self.prediction = prediction  # L2, L4 y L13: guarda la clase que devolvería esta hoja.
        self.attribute = attribute  # L15: atributo de decisión; None indica que el nodo es una hoja.
        self.threshold = threshold  # Campo opcional de los auxiliares; el entrenamiento literal no usa cortes.
        self.children = {}  # L19: almacena las ramas y sus subárboles hijos.


## 3. Impurity evaluation

Python identifiers cannot contain a hyphen, so the pseudocode's
`impurityEval-1(D)` and `impurityEval-2(Ai, D)` become `impurityEval_1`
and `impurityEval_2`.

- **Before splitting:** `p0 = H(D) = -sum(p(c) * log2(p(c)))`.
- **After splitting:** `pi = sum(|Dj| / |D| * H(Dj))`.
- **Impurity reduction / information gain:** `p0 - pi`.
- **Split information:** `-sum(qj * log2(qj))`, where `qj = |Dj| / |D|`.
- **Gain Ratio:** information gain divided by split information. This normalizes
  by branch proportions, not class proportions; a zero denominator gives score zero.

Empty datasets have zero entropy in these helpers. A constant attribute has
zero information gain and zero split information. All logarithms use base 2.
The existing optional `threshold` argument of
`partition()` and `impurityEval_2()` is a standalone utility;
`decision_tree()` never passes a numerical threshold.


In [8]:
from collections import Counter  # Usa Python estándar para contar las clases de D.
from math import log2  # Usa el logaritmo en base 2 para calcular entropía.


def impurityEval_1(D):  # L7: define impurityEval-1(D), la impureza antes de dividir.
    if not D:  # Caso auxiliar: comprueba si el conjunto de datos está vacío.
        return 0.0  # Asigna impureza cero al conjunto vacío.
    counts = Counter(row[-1] for row in D)  # Cuenta las clases de la última columna para obtener sus probabilidades.
    return -sum((count / len(D)) * log2(count / len(D))  # L7: calcula H(D) = -suma de p(c) por log2(p(c)).
                for count in counts.values())  # Incluye en la entropía cada clase presente en D.


def partition(Ai, D, threshold=None):  # L16: define cómo dividir D según los valores del atributo Ai.
    groups = {}  # Prepara los subconjuntos Dj, uno por rama.
    for row in D:  # Recorre las observaciones para asignarlas a su subconjunto.
        branch = row[Ai] if threshold is None else row[Ai] <= threshold  # L16: usa una categoría o, para datos numéricos, la comparación con el umbral.
        groups.setdefault(branch, []).append(row)  # L16: agrega la observación al subconjunto Dj que le corresponde.
    return groups  # Devuelve las particiones no vacías para evaluar o entrenar los hijos.


def impurityEval_2(Ai, D, threshold=None):  # L9: define impurityEval-2(Ai, D), la impureza después de dividir.
    if not D:  # Caso auxiliar: comprueba si el conjunto de datos está vacío.
        return 0.0  # Asigna impureza cero al conjunto vacío.
    groups = partition(Ai, D, threshold)  # L16: obtiene los subconjuntos necesarios para evaluar esta división.
    return sum(len(group) / len(D) * impurityEval_1(group)  # L9: suma la entropía de cada subconjunto ponderada por su tamaño.
               for group in groups.values())  # Considera todos los subconjuntos producidos por Ai.


def split_information(Ai, D):
    if not D:
        return 0.0
    groups = partition(Ai, D)  # Only observed values produce nonempty branches.
    return -sum((len(group) / len(D)) * log2(len(group) / len(D))
                for group in groups.values())


## 4. Train the tree

`decision_tree()` returns the tree `T` with this signature:

```python
decision_tree(D, A, min_gain=1e-12, max_depth=None,
              min_samples_split=2, depth=0, n_total=None)
```

Information Gain is active in the attribute loop: `score = gain`. To switch,
comment that assignment and uncomment the adjacent Gain Ratio assignment to
the same `score` variable, then rerun the five code cells from top to bottom.
No other code or function arguments need changing.

### Stopping controls

Edit `max_depth=None` and `min_samples_split=2` in the training call at the
bottom of this code cell, for example `max_depth=3, min_samples_split=5`.
Rerun the training and evaluation cells after changing these values.

- The root has depth **0**. `max_depth=None` imposes no explicit depth limit;
  a node at `depth >= max_depth` becomes a majority-class leaf. Thus zero
  produces a root majority leaf, and one allows only the root to split.
- `min_samples_split` stops a node when `len(D) < min_samples_split`; equality
  permits splitting. This uses each node's current data, not the root count,
  and is **not** a minimum leaf size: a split can create smaller children.
- `depth=0` is recursion bookkeeping, not a tuning control. Each child receives
  `depth + 1` and the same maximum depth and sample minimum.
- `n_total=None` records the original training count once from the initial `D`.
  Every child receives that same total; it is not reset to a parent/child count.

Either limit can stop training before attribute scoring. The leaf predicts
the current node's majority class, with the existing first-encountered tie rule.
All six previous positional arguments remain compatible; `n_total` is appended.

Its comments reference the supplied image's line numbers:

1. **L1-L2:** if all examples have the same class, return a leaf with that class.
2. **L3-L4:** if no attributes remain, return a majority-class leaf.
3. **L7-L11:** compute `p0` and each attribute's `pi`; select the greatest
   active score (Information Gain or Gain Ratio). A tie uses the first attribute in `A`.
4. **L12-L13:** compute `weighted_gain = len(D) / n_total * (p0 - pg)` for
   the selected attribute. If `weighted_gain < min_gain`, return a majority-class
   leaf; equality still permits a split. This minimum uses **root-weighted
   Information Gain**, never the Gain Ratio ranking score.
5. **L15-L18:** create a decision node and partition by each observed value;
   only nonempty subsets become branches.
6. **L19-L20:** recurse on each branch with the selected attribute removed,
   regardless of its value type. Forward `min_gain`, `max_depth`, and
   `min_samples_split` and original `n_total` unchanged, and increase `depth` by one.

Each recursive call removes an attribute. With `min_gain=0`, even a constant
attribute can create one branch before being removed when the depth/sample
controls permit it, exactly as the image's strict comparison allows.
The default remains `1e-12`.
The minimum weights local entropy reduction by this node's fraction of the
original training data. The root factor is 1; descendants use the fixed original
total, not their parent's size. It is never divided by split information.
Attribute ranking still uses local Information Gain or local Gain Ratio.
Selection happens first; if that attribute fails the minimum, return a leaf
rather than substituting another attribute.

This follows the threshold weighting documented for scikit-learn's
[`min_impurity_decrease`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html):

```text
N_t / N * (impurity - N_t_R / N_t * right_impurity
                   - N_t_L / N_t * left_impurity)
```

Here `N` is original `n_total`, `N_t` is `len(D)`, and `impurityEval_2()`
generalizes the bracket to all observed branches using conditional entropy.
All rows have unit weight; no sample weights are implemented. This aligns only
threshold weighting, not full DT/RF behavior: impurity criteria, bootstrap,
encoding, and split topology can still differ.

There is no additional C4.5 gain filter, numerical cut search, pruning, or input-validation
layer inside training. Majority counting, impurity evaluation, partitioning,
prediction, and display are separate functions; there is no nested builder.

`predict_one()` and `print_tree()` are helpers outside the training algorithm.
Prediction rows contain features in training-column order, without the class.


In [9]:
def majority_class(D):  # Auxiliar: obtiene la clase más frecuente para las hojas y el respaldo de predicción.
    return Counter(row[-1] for row in D).most_common(1)[0][0]  # En un empate, conserva la primera clase observada.


def decision_tree(D, A, min_gain=1e-12, max_depth=None,
                  min_samples_split=2, depth=0, n_total=None):  # Track depth and original training count.
    if n_total is None:
        n_total = len(D)  # Initialize once per independent training call.
    A = list(A)  # Conserva una lista local de los atributos disponibles.
    classes = {row[-1] for row in D}  # L1: obtiene las clases presentes en D.
    if len(classes) == 1:  # L1: todos los ejemplos pertenecen a la misma clase.
        return Node(D[0][-1])  # L2: crea una hoja etiquetada con esa clase.
    elif not A:  # L3: no quedan atributos para dividir D.
        return Node(majority_class(D))  # L4: crea una hoja con la clase más frecuente.
    elif max_depth is not None and depth >= max_depth:  # The root starts at depth zero.
        return Node(majority_class(D))
    elif len(D) < min_samples_split:  # Equality still permits a split.
        return Node(majority_class(D))
    else:  # L5-L6: D mezcla clases y todavía hay atributos disponibles.
        p0 = impurityEval_1(D)  # L7: calcula la impureza inicial de D.
        scores = []  # Keep raw impurity and ranking score separate.
        for Ai in A:  # L8: recorre los atributos disponibles.
            pi = impurityEval_2(Ai, D)  # L9: calcula la impureza después de dividir por Ai.
            gain = p0 - pi  # Raw Information Gain at the current node.
            split_info = split_information(Ai, D)
            score = gain  # Information Gain
            # score = gain / split_info if split_info else 0.0  # Gain Ratio
            scores.append((Ai, pi, score))  # L10: retain both quantities for the winner.
        Ag, pg, _ = max(scores, key=lambda item: item[2])  # L11: first attribute wins ties.
        weighted_gain = len(D) / n_total * (p0 - pg)
        if weighted_gain < min_gain:  # L12: root-weighted Information Gain, never Gain Ratio.
            return Node(majority_class(D))  # L13: crea una hoja con la clase más frecuente.
        else:  # L14: la reducción alcanza el umbral.
            node = Node(majority_class(D), attribute=Ag)  # L15: crea T como nodo de decisión sobre Ag.
            groups = partition(Ag, D)  # L16: divide D según los valores observados de Ag.
            remaining = [attribute for attribute in A if attribute != Ag]  # L20: forma A menos Ag.
            for branch, group in groups.items():  # L17: recorre los subconjuntos Dj.
                if group:  # L18: solo crea ramas para subconjuntos no vacíos.
                    node.children[branch] = decision_tree(
                        group, remaining, min_gain,
                        max_depth=max_depth,
                        min_samples_split=min_samples_split,
                        depth=depth + 1,
                        n_total=n_total,
                    )  # L19-L20: grow Tj with the same limits at the next depth.
            return node  # L21-L24: devuelve el árbol construido.


def predict_one(tree, row):  # Paso adicional: clasifica un dato utilizando el árbol ya entrenado.
    node = tree  # Comienza la predicción en la raíz T.
    while node.attribute is not None:  # Recorre los nodos de decisión hasta encontrar una hoja.
        value = row[node.attribute]  # Lee el valor del atributo elegido en el nodo actual.
        branch = value if node.threshold is None else value <= node.threshold  # Selecciona la rama categórica o el resultado del corte numérico.
        if branch not in node.children:  # Detecta una categoría que no apareció al entrenar este nodo.
            return node.prediction  # Devuelve la clase guardada: la de la hoja o la mayoritaria de respaldo.
        node = node.children[branch]  # Desciende al subárbol hijo correspondiente al dato.
    return node.prediction  # Devuelve la clase guardada: la de la hoja o la mayoritaria de respaldo.


def print_tree(node, columns, indent=""):  # Paso adicional: muestra el árbol entrenado sin bibliotecas de visualización.
    if node.attribute is None:  # Reconoce una hoja como las creadas en L2, L4 o L13.
        print(f"{indent}Class: {node.prediction}")  # Muestra la clase cj asignada a la hoja.
        return  # Termina la visualización de esta hoja.
    print(f"{indent}{columns[node.attribute]}")  # Muestra el nombre del atributo Ag del nodo de decisión.
    for branch, child in node.children.items():  # Recorre las ramas y los subárboles creados en L19-L20.
        label = repr(branch)  # Prepara la etiqueta del valor categórico vj.
        if node.threshold is not None:  # Comprueba si la rama pertenece a una división numérica.
            label = f"{'<=' if branch else '>'} {node.threshold:g}"  # Muestra qué lado del umbral representa esta rama.
        print(f"{indent}  [{label}]")  # Imprime la condición de la rama.
        print_tree(child, columns, indent + "    ")  # Muestra recursivamente el subárbol hijo Tj con mayor sangría.


tree = decision_tree(D, A, max_depth=None, min_samples_split=2)  # Invoca el algoritmo con los datos y atributos para obtener el árbol entrenado.
print_tree(tree, columns)  # Muestra el árbol resultante al finalizar el cuarto bloque.


Outlook
  ['sunny']
    Humidity
      ['high']
        Class: no
      ['normal']
        Class: yes
  ['overcast']
    Class: yes
  ['rain']
    Wind
      ['weak']
        Class: yes
      ['strong']
        Class: no


## 5. Test data, accuracy, and predictions

These six labeled observations are distinct from the training rows. The trained
tree is reused without retraining. Accuracy is the fraction of correct class
predictions; it is the only reported evaluation metric. This tiny, hand-labeled
example demonstrates execution, not an estimate of real-world generalization.
The existing output also lists the predicted classes after accuracy.


In [10]:
test_matrix = [  
    columns.copy(),  
    ["sunny", "cool", "high", "strong", "no"],  
    ["sunny", "mild", "normal", "weak", "yes"],  
    ["overcast", "hot", "high", "strong", "yes"],  
    ["rain", "hot", "normal", "weak", "yes"],  
    ["rain", "hot", "high", "strong", "no"],  
    ["sunny", "hot", "normal", "strong", "yes"],  
] 

test_rows = test_matrix[1:]  # Excluye la fila de encabezados antes de evaluar el modelo.
predictions = [predict_one(tree, row[:-1]) for row in test_rows]  # Clasifica cada ejemplo sin pasar su clase real ni volver a entrenar.
accuracy = sum(prediction == row[-1]  # Cuenta las predicciones que coinciden con la clase esperada.
               for prediction, row in zip(predictions, test_rows)) / len(test_rows)  # Divide los aciertos por la cantidad de ejemplos: accuracy.
print(f"Accuracy: {accuracy:.2%}")  # Muestra únicamente la métrica solicitada, expresada como porcentaje.
print (predictions)


Accuracy: 100.00%
['no', 'yes', 'yes', 'yes', 'no', 'yes']
